# CAPA

`CAPA` (Collective And Point Anomalies) jointly detects collective (segment) anomalies and point anomalies against a fixed baseline. Runtime is linear when there are many anomalies, which makes it the fast alternative to [CircularBinarySegmentation](circular_binseg.ipynb) whenever the baseline is well-defined and stable.

## Basic usage

The example below plants two short anomalous segments in an otherwise stationary series and recovers them with [L2Saving](../../api_reference/auto_generated/skchange.new_api.interval_scorers.L2Saving.rst) as the segment saving.

In [ ]:
import plotly.io as pio

from skchange.new_api.datasets import generate_piecewise_normal_data
from skchange.new_api.detectors import CAPA
from skchange.new_api.interval_scorers import L2Saving
from skchange.new_api.utils.plotting import plot_detections

pio.renderers.default = "notebook"

X = generate_piecewise_normal_data(
    means=[0, 5, 0, -4, 0],
    lengths=[40, 10, 60, 5, 35],
    seed=1,
)

detector = CAPA(L2Saving(), min_segment_length=3, penalty_scale=1.0)
detector.fit(X)
anomalies = detector.predict_segment_anomalies(X)

plot_detections(X, segment_anomalies=anomalies).show()
print(anomalies)

## Adding point anomalies

Setting `include_point_anomalies=True` and providing a `point_saving` lets `CAPA` co-detect isolated point outliers alongside segment anomalies. `predict_all` returns them separately if you need to distinguish.

In [ ]:
detector = CAPA(
    segment_saving=L2Saving(),
    point_saving=L2Saving(),
    min_segment_length=3,
    include_point_anomalies=True,
)
result = detector.fit(X).predict_all(X)

print("Segment anomalies:", result["segment_anomalies"])
print("Point anomalies:  ", result["point_anomalies"])

## Parameters worth knowing

- `segment_saving`: an interval scorer of type `saving` used for collective anomalies.
- `point_saving`: optional interval scorer of type `saving` used for point anomalies. Required when `include_point_anomalies=True`.
- `segment_penalty`, `point_penalty`, `penalty_scale`: thresholds applied to segment and point savings. Larger values produce fewer detections. Scalar or array; see [Penalties](../concepts/penalties.ipynb).
- `min_segment_length`, `max_segment_length`: bounds on the length of segment anomalies.
- `include_point_anomalies`: whether to jointly detect point anomalies.

See the full API reference for [CAPA](../../api_reference/auto_generated/skchange.new_api.detectors.CAPA.rst).

## See also

- [CircularBinarySegmentation](circular_binseg.ipynb): a more flexible transient-change detector when the baseline isn't fixed, at higher computational cost.
- [Change detectors](../concepts/change_detectors.ipynb): background on the different search strategies.